In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import random
import shutil
from pathlib import Path
from tqdm import tqdm

# =========================================================
# SUGGESTED TARGET
# =========================================================
TARGET_PER_CLASS = 5000
SEED = 42
random.seed(SEED)

# =========================================================
# OUTPUT
# =========================================================
DEST_ROOT = Path("/kaggle/working/KidneyBalancedDataset")
NORMAL_DEST = DEST_ROOT / "normal"
DEFECTED_DEST = DEST_ROOT / "defected"

# =========================================================
# SOURCE PATHS
# =========================================================
NORMAL_DIR_1 = Path(
    "/kaggle/input/datasets/gurjeetkaurmangat/"
    "kidney-ultrasound-images-stone-and-no-stone/"
    "my dataset final 512x512(implemented)/Normal"
)

NORMAL_DIR_2 = Path("/kaggle/working/Kidney/normal")

DISEASE_ROOT = Path(
    "/kaggle/input/datasets/ahmedshayan1/dataset-multicalasess/"
    "KidneyDiseaseClassification.folder/train"
)

CYSTS_DIR = DISEASE_ROOT / "cysts"
STONES_DIR = DISEASE_ROOT / "stones"
TUMORS_DIR = DISEASE_ROOT / "tumors"

STONE_DIR_2 = Path(
    "/kaggle/input/datasets/gurjeetkaurmangat/"
    "kidney-ultrasound-images-stone-and-no-stone/"
    "my dataset final 512x512(implemented)/stone"
)

# =========================================================
# IMAGE EXTENSIONS
# =========================================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# =========================================================
# HELPERS
# =========================================================
def get_images(folder: Path):
    if not folder.exists():
        return []
    return [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]

def clean_folder(folder: Path):
    folder.mkdir(parents=True, exist_ok=True)
    for f in folder.iterdir():
        if f.is_file():
            f.unlink()

def proportional_counts(source_counts: dict, target_total: int):
    total_available = sum(source_counts.values())
    raw = {name: (count / total_available) * target_total for name, count in source_counts.items()}
    picked = {name: int(v) for name, v in raw.items()}

    assigned = sum(picked.values())
    remaining = target_total - assigned

    remainders = sorted(
        [(name, raw[name] - picked[name]) for name in picked],
        key=lambda x: x[1],
        reverse=True
    )

    idx = 0
    while remaining > 0:
        name = remainders[idx % len(remainders)][0]
        if picked[name] < source_counts[name]:
            picked[name] += 1
            remaining -= 1
        idx += 1

    return picked

def copy_files(file_tuples, dest_dir: Path, prefix: str):
    dest_dir.mkdir(parents=True, exist_ok=True)
    copied = 0

    for i, (src_name, src_path) in enumerate(tqdm(file_tuples, desc=f"Copying -> {dest_dir.name}"), start=1):
        dst_name = f"{prefix}_{src_name}_{i}_{src_path.name}"
        dst_path = dest_dir / dst_name

        counter = 1
        while dst_path.exists():
            dst_path = dest_dir / f"{prefix}_{src_name}_{i}_{counter}_{src_path.name}"
            counter += 1

        shutil.copy2(src_path, dst_path)
        copied += 1

    return copied

# =========================================================
# LOAD IMAGES
# =========================================================
normal_sources = {
    "normal_dataset": get_images(NORMAL_DIR_1),
    "normal_working": get_images(NORMAL_DIR_2),
}

defected_sources = {
    "cysts": get_images(CYSTS_DIR),
    "stones_multi": get_images(STONES_DIR),
    "tumors": get_images(TUMORS_DIR),
    "stones_dataset": get_images(STONE_DIR_2),
}

# =========================================================
# COUNTS
# =========================================================
normal_counts = {name: len(imgs) for name, imgs in normal_sources.items()}
defected_counts = {name: len(imgs) for name, imgs in defected_sources.items()}

normal_total = sum(normal_counts.values())
defected_total = sum(defected_counts.values())

print("===== SOURCE COUNTS =====")
for name, count in normal_counts.items():
    print(f"{name:15s}: {count}")
for name, count in defected_counts.items():
    print(f"{name:15s}: {count}")

print(f"\nTotal normal available   : {normal_total}")
print(f"Total defected available : {defected_total}")

final_target = min(TARGET_PER_CLASS, normal_total, defected_total)
print(f"\nSuggested target per class: {TARGET_PER_CLASS}")
print(f"Final target per class    : {final_target}")

# =========================================================
# PRINT PROPORTIONS
# =========================================================
print("\n===== NORMAL PROPORTIONS =====")
for name, count in normal_counts.items():
    pct = (count / normal_total) * 100
    print(f"{name:15s}: {count:4d} ({pct:.2f}%)")

print("\n===== DEFECTED PROPORTIONS =====")
for name, count in defected_counts.items():
    pct = (count / defected_total) * 100
    print(f"{name:15s}: {count:4d} ({pct:.2f}%)")

# =========================================================
# PICK COUNTS
# =========================================================
normal_pick_counts = proportional_counts(normal_counts, final_target)
defected_pick_counts = proportional_counts(defected_counts, final_target)

print("\n===== NORMAL PICK COUNTS =====")
for name, count in normal_pick_counts.items():
    print(f"{name:15s}: {count}")

print("\n===== DEFECTED PICK COUNTS =====")
for name, count in defected_pick_counts.items():
    print(f"{name:15s}: {count}")

# =========================================================
# SAMPLE FILES
# =========================================================
selected_normal = []
for name, imgs in normal_sources.items():
    chosen = random.sample(imgs, normal_pick_counts[name])
    selected_normal.extend([(name, p) for p in chosen])

selected_defected = []
for name, imgs in defected_sources.items():
    chosen = random.sample(imgs, defected_pick_counts[name])
    selected_defected.extend([(name, p) for p in chosen])

# =========================================================
# PREPARE DESTINATION
# =========================================================
clean_folder(NORMAL_DEST)
clean_folder(DEFECTED_DEST)

# =========================================================
# COPY
# =========================================================
copied_normal = copy_files(selected_normal, NORMAL_DEST, "normal")
copied_defected = copy_files(selected_defected, DEFECTED_DEST, "defected")

# =========================================================
# FINAL STATS
# =========================================================
print("\n===== FINAL DATASET =====")
print(f"Saved normal   : {copied_normal}")
print(f"Saved defected : {copied_defected}")
print(f"Dataset path   : {DEST_ROOT}")

In [69]:


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
 
# ============================================================================
# CBAM (Convolutional Block Attention Module)
# ============================================================================
 
class ChannelAttention(nn.Module):
    """Channel Attention Module"""
    def __init__(self, channels, reduction=8):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
 
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)
 
 
class SpatialAttention(nn.Module):
    """Spatial Attention Module"""
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = 3 if kernel_size == 7 else 1
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()
 
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)
 
 
class CBAM(nn.Module):
    """CBAM Attention Module"""
    def __init__(self, channels, reduction=8, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
 
    def forward(self, x):
        out = x * self.channel_attention(x)
        out = out * self.spatial_attention(out)
        return out
 
 
# ============================================================================
# Bottleneck Block with CBAM
# ============================================================================
 
class Bottleneck(nn.Module):
    expansion = 4
 
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, 
                               kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride
 
    def forward(self, x):
        identity = x
 
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
 
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
 
        out = self.conv3(out)
        out = self.bn3(out)
 
        if self.downsample is not None:
            identity = self.downsample(x)
 
        out += identity
        out = self.relu(out)
 
        return out
 
 
# ============================================================================
# ResNet50 with CBAM Attention
# ============================================================================
 
class ResNet50_CBAM(nn.Module):
    """ResNet50 with CBAM attention modules after each residual block"""
    
    def __init__(self, num_classes=1000):
        super(ResNet50_CBAM, self).__init__()
        self.in_channels = 64
 
        # Initial convolution layer
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
 
        # Residual layers with CBAM
        self.layer1 = self._make_layer(Bottleneck, 64, 3, stride=1)
        self.cbam1 = CBAM(256)
        
        self.layer2 = self._make_layer(Bottleneck, 128, 4, stride=2)
        self.cbam2 = CBAM(512)
        
        self.layer3 = self._make_layer(Bottleneck, 256, 6, stride=2)
        self.cbam3 = CBAM(1024)
        
        # self.layer4 = self._make_layer(Bottleneck, 512, 3, stride=2)
        # self.cbam4 = CBAM(2048)

        # self.layer5 = self._make_layer(Bottleneck, 1024, 3, stride=2)
        # self.cbam5 = CBAM(4096)

        # self.layer6 = self._make_layer(Bottleneck, 2048, 3, stride=2)
        # self.cbam6 = CBAM(8192)

        # self.refine = nn.Sequential(
        # nn.Conv2d(2048, 1024, 1),
        # nn.BatchNorm2d(1024),
        # nn.ReLU(inplace=True),
        # nn.Conv2d(1024, 2048, 1),
        # )
        
        # Global average pooling and fully connected
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(1024, num_classes)
 
        # Initialize weights
        self._init_weights()
 
    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion, 
                         kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )
 
        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))
 
        return nn.Sequential(*layers)
 
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
 
    def forward(self, x):
        # Initial convolution
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
 
        # Layer 1 + CBAM
        x = self.layer1(x)
        x = self.cbam1(x)
 
        # Layer 2 + CBAM
        x = self.layer2(x)
        x = self.cbam2(x)
 
        # Layer 3 + CBAM
        x = self.layer3(x)
        x = self.cbam3(x)
 
        # # Layer 4 + CBAM
        # x = self.layer4(x)
        # x = self.cbam4(x)

        # # Layer 5 + CBAM
        # x = self.layer5(x)
        # x = self.cbam5(x)

        # # Layer 6 + CBAM
        # x = self.layer6(x)
        # x = self.cbam6(x)
 
        # Global average pooling and classification
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
 
        return x
 


In [35]:
model = ResNet50_CBAM(num_classes=1000)

In [36]:
model

ResNet50_CBAM(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stri

In [37]:
print("hellow")

hellow


In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomResizedCrop(224),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dataset = ImageFolder(
    "/kaggle/working/KidneyBalancedDataset",
    transform=train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

test_dataset = ImageFolder(
    "/kaggle/working/Test_unseen",
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

model = ResNet50_CBAM(num_classes=2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total
    avg_loss = running_loss / len(loader)

    return acc, avg_loss


num_epochs = 50

best_test_acc = 0.0
save_path = "best_model.pth"

for epoch in range(num_epochs):
    model.train()

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loop.set_postfix(loss=loss.item())

    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader)

    # =========================
    # 🔴 TEST ON UNSEEN DATA
    # =========================
    test_acc, test_loss = evaluate(model, test_loader)

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")
    print("-" * 50)

    # =========================
    # 💾 SAVE BEST MODEL
    # =========================
    if test_acc > best_test_acc:
        best_test_acc = test_acc

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "test_acc": test_acc,
            "train_acc": train_acc
        }, save_path)

        print(f"🔥 BEST MODEL SAVED | Test Acc: {test_acc:.2f}%")



Device: cuda


Epoch 1/50: 100%|██████████| 552/552 [00:48<00:00, 11.32it/s, loss=0.579]



Epoch [1/50]
Train Loss: 0.6397 | Train Acc: 62.17%
Test Loss: 1.1220 | Test Acc: 55.56%
--------------------------------------------------
🔥 BEST MODEL SAVED | Test Acc: 55.56%


Epoch 2/50: 100%|██████████| 552/552 [00:48<00:00, 11.34it/s, loss=0.482]



Epoch [2/50]
Train Loss: 0.4999 | Train Acc: 75.40%
Test Loss: 1.3916 | Test Acc: 50.00%
--------------------------------------------------


Epoch 3/50: 100%|██████████| 552/552 [00:48<00:00, 11.34it/s, loss=0.593]


... (91 outputs left out)

Epoch 49/50: 100%|██████████| 552/552 [00:48<00:00, 11.33it/s, loss=0.0921]



Epoch [49/50]
Train Loss: 0.2173 | Train Acc: 91.27%
Test Loss: 5.3452 | Test Acc: 55.56%
--------------------------------------------------


Epoch 50/50: 100%|██████████| 552/552 [00:48<00:00, 11.34it/s, loss=0.0714]



Epoch [50/50]
Train Loss: 0.2258 | Train Acc: 90.51%
Test Loss: 5.2066 | Test Acc: 55.56%
--------------------------------------------------


In [85]:
model = ResNet50_CBAM(num_classes=2)

checkpoint = torch.load("best_model.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
class_names = ["Normal", "Abnormal"]  # change if needed
from PIL import Image
import os

folder_path = "/kaggle/input/datasets/ahmedshayan1/normal"

import torch.nn.functional as F

for img_name in os.listdir(folder_path):
    img_path = os.path.join(folder_path, img_name)

    try:
        image = Image.open(img_path).convert("L")
        image = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(image)

            # 🔥 Convert logits → probabilities
            probs = F.softmax(outputs, dim=1)

            # Get predicted class
            conf, pred = torch.max(probs, 1)

        predicted_class = class_names[pred.item()]
        confidence = conf.item()

        # Print both class probabilities
        prob_list = probs.squeeze().cpu().numpy()

        print(f"{img_name} → {predicted_class}")
        print(f"Confidence: {confidence:.4f}")
        print(f"Probabilities: {dict(zip(class_names, prob_list))}")
        print("-" * 50)

    except Exception as e:
        print(f"Error processing {img_name}: {e}")



Tes3.png → Normal
Confidence: 1.0000
Probabilities: {'Normal': np.float32(0.9999999), 'Abnormal': np.float32(1.1202034e-07)}
--------------------------------------------------
Normal.jpeg → Normal
Confidence: 1.0000
Probabilities: {'Normal': np.float32(0.99998343), 'Abnormal': np.float32(1.657553e-05)}
--------------------------------------------------
Screenshot 2026-04-11 at 11.11.48PM.png → Normal
Confidence: 1.0000
Probabilities: {'Normal': np.float32(0.99999833), 'Abnormal': np.float32(1.7225211e-06)}
--------------------------------------------------


##### Final Pipeline

In [117]:
model = ResNet50_CBAM(num_classes=2)

checkpoint = torch.load("best_model.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

class_names = ["Normal", "Abnormal"]

from PIL import Image
import os
import torch
import torch.nn.functional as F

folder_path = "/kaggle/input/datasets/ahmedshayan1/narmal"

for img_name in os.listdir(folder_path):
    img_path = os.path.join(folder_path, img_name)

    try:
        image = Image.open(img_path).convert("L")
        image = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(image)
            probs = F.softmax(outputs, dim=1)

        probs = probs.squeeze().cpu().numpy()

        normal_prob = probs[0]
        abnormal_prob = probs[1]

        # =========================
        # 🔥 YOUR CUSTOM RULE
        # =========================

        if normal_prob >= 0.999999:
            final_label = "Normal"   
        else:
            if abnormal_prob > 0.00009:
                final_label = "Disease"
            else:
                final_label = "Normal"

        print(f"{img_name}")
        print(f"Predicted: {final_label}")
        print(f"Normal Prob   : {normal_prob:.6f}")
        print(f"Abnormal Prob : {abnormal_prob:.6f}")
        print("-" * 50)

    except Exception as e:
        print(f"Error processing {img_name}: {e}")

Normal_123.jpeg
Predicted: Disease
Normal Prob   : 0.999800
Abnormal Prob : 0.000200
--------------------------------------------------


In [76]:
import torch
import torch.nn.functional as F
from PIL import Image
import os
from tqdm import tqdm
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# TRANSFORM
# =========================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# =========================
# LOAD MODEL (already done by you)
# =========================
model.eval()

# =========================
# FOLDER PATH (single class folder)
# =========================
folder_path = "/kaggle/input/datasets/ahmedshayan1/disease"

image_files = [
    f for f in os.listdir(folder_path)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

print("Total images:", len(image_files))

# =========================
# INFERENCE
# =========================
class_names = ["Class0", "Class1"]  # change if needed

print("\n--- Predictions with Probabilities ---\n")

with torch.no_grad():
    for i, img_name in enumerate(tqdm(image_files)):

        img_path = os.path.join(folder_path, img_name)

        image = Image.open(img_path).convert("RGB")
        image = transform(image).unsqueeze(0).to(device)

        outputs = model(image)
        probs = F.softmax(outputs, dim=1)

        prob_values = probs.cpu().numpy()[0]
        pred_class = torch.argmax(probs, dim=1).item()

        print(f"Image {i+1}: {img_name}")
        print(f"Predicted Class : {class_names[pred_class]}")
        print(f"Probabilities   : {prob_values}")
        print("-" * 50)

Total images: 3

--- Predictions with Probabilities ---



100%|██████████| 3/3 [00:00<00:00, 39.66it/s]

Image 1: Screenshot 2026-04-11 at 11.18.03PM.png
Predicted Class : Class0
Probabilities   : [9.9985671e-01 1.4324034e-04]
--------------------------------------------------
Image 2: Screenshot 2026-04-11 at 11.18.19PM.png
Predicted Class : Class0
Probabilities   : [0.99536633 0.00463371]
--------------------------------------------------
Image 3: Screenshot 2026-04-11 at 11.18.28PM.png
Predicted Class : Class0
Probabilities   : [0.9923046  0.00769541]
--------------------------------------------------


In [ ]:
print("hellow")

In [9]:
import os
import shutil
import hashlib
from tqdm import tqdm

# Input paths
input_base = "/kaggle/input/datasets/ahmedshayan1/test-unseen/Test"
output_base = "/kaggle/working/Test_unseen"

classes = ["Diseased", "Normal"]

# Create output folders
for c in classes:
    os.makedirs(os.path.join(output_base, c), exist_ok=True)

def get_md5(file_path):
    """Return MD5 hash of a file"""
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

seen_hashes = set()

total_files = 0
for c in classes:
    total_files += len(os.listdir(os.path.join(input_base, c)))

print("Total files found:", total_files)

for c in classes:
    src_dir = os.path.join(input_base, c)
    dst_dir = os.path.join(output_base, c)

    for img_name in tqdm(os.listdir(src_dir), desc=f"Processing {c}"):
        src_path = os.path.join(src_dir, img_name)

        # skip non-files
        if not os.path.isfile(src_path):
            continue

        try:
            file_hash = get_md5(src_path)
        except:
            print("Skipping unreadable file:", src_path)
            continue

        # If duplicate → skip
        if file_hash in seen_hashes:
            continue

        seen_hashes.add(file_hash)

        # Copy unique file
        dst_path = os.path.join(dst_dir, img_name)
        shutil.copy2(src_path, dst_path)

print("\nDone! Clean dataset saved at:", output_base)
print("Total unique images:", len(seen_hashes))

Total files found: 20


Processing Normal: 100%|██████████| 8/8 [00:00<00:00, 448.98it/s]


Done! Clean dataset saved at: /kaggle/working/Test_unseen
Total unique images: 18


In [ ]:
!rm -r "/kaggle/working/Test_unseen/Normal/images.jpg"